[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/56_adamw_solution.ipynb)

# Solution: AdamW (Decoupled Weight Decay)

Reference solution.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

class MyAdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0
    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_()
                p.grad.zero_()
    
    @torch.no_grad()
    def step(self):
        self.t += 1
        bc1 = 1 - self.beta1 ** self.t
        bc2 = 1 - self.beta2 ** self.t
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            # Decoupled weight decay — gradient じゃなく PARAM に適用
            if self.weight_decay != 0:
                p.mul_(1 - self.lr * self.weight_decay)
            g = p.grad
            m.mul_(self.beta1).add_(g, alpha=1 - self.beta1)
            v.mul_(self.beta2).addcmul_(g, g, value=1 - self.beta2)
            m_hat = m / bc1
            v_hat = v / bc2
            p.addcdiv_(m_hat, v_hat.sqrt() + self.eps, value=-self.lr)

In [ ]:
import torch
torch.manual_seed(0)
p = torch.randn(5, requires_grad=True)
opt = MyAdamW([p], lr=0.01, weight_decay=0.1)
for step in range(3):
    loss = (p ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'step {step}: |p| = {p.norm().item():.4f}')

In [ ]:
from torch_judge import check
check("adamw")